# 02 · Predicción de recompra

Ruta de continuación con las seis fases CRISP-DM. Los CSV públicos se cargan desde el repositorio; no se descargan datos manualmente.

## 1. Comprensión del negocio

Estimaremos si un cliente con compras históricas vuelve a comprar durante los 90 días siguientes a una fecha de corte. Es un ejercicio para explorar retención, no una garantía de comportamiento ni una instrucción para tomar decisiones sobre personas.

## 2. Comprensión de los datos

Usamos pedidos, clientes, líneas de pedido y productos. `customer_unique_id` agrupa pedidos históricos; no se usa como característica ni se presenta en la interfaz. Consulta el [diccionario de datos](../docs/diccionario_datos.md).


In [ ]:
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_BASE_URL = "https://raw.githubusercontent.com/JuanParias29/ML_MiniWorkshop/main/data"

def load_dataset_csv(filename):
    url = f"{DATA_BASE_URL}/{filename}"
    try:
        return pd.read_csv(url)
    except (HTTPError, URLError) as error:
        raise RuntimeError(
            f"No se pudo cargar {filename} desde {url}. Compruebe la conexión a internet "
            "y confirme que el CSV está publicado en la carpeta main/data del repositorio."
        ) from error
orders = load_dataset_csv("olist_orders_dataset.csv")
customers = load_dataset_csv("olist_customers_dataset.csv")
items = load_dataset_csv("olist_order_items_dataset.csv")
products = load_dataset_csv("olist_products_dataset.csv")
print(f"Pedidos cargados: {len(orders):,}; grupos de clientes seudónimos: {customers['customer_unique_id'].nunique():,}")


## 3. Preparación de los datos

La observación cubre los 180 días antes del corte. La etiqueta vale 1 si hay un pedido entregado en los 90 días posteriores. Separamos las ventanas por fecha para que los predictores solo usen información anterior al corte y la etiqueta solo use información futura.


In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)
orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner"
)
delivered = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_purchase_timestamp"].notna()
].copy()
if delivered.empty:
    raise ValueError("No hay pedidos entregados con fecha válida para formar la etiqueta.")
window_days = 90
history_days = 180
cutoff = delivered["order_purchase_timestamp"].max() - pd.Timedelta(days=window_days)
history_start = cutoff - pd.Timedelta(days=history_days)
future_end = cutoff + pd.Timedelta(days=window_days)
observed_orders = delivered.loc[
    delivered["order_purchase_timestamp"].ge(history_start)
    & delivered["order_purchase_timestamp"].lt(cutoff)
].copy()
future_orders = delivered.loc[
    delivered["order_purchase_timestamp"].ge(cutoff)
    & delivered["order_purchase_timestamp"].le(future_end)
]
print(f"Inicio observación: {history_start.date()}; corte exclusivo: {cutoff.date()}")
print(f"Ventana etiqueta: [{cutoff.date()}, {future_end.date()}]")
print(f"Pedidos históricos: {len(observed_orders):,}; futuros: {len(future_orders):,}")

Las características se agregan exclusivamente con pedidos entregados anteriores al corte. El resultado futuro se convierte en una etiqueta binaria y no participa en frecuencia, gasto ni variedad.


In [ ]:
if observed_orders.empty:
    raise ValueError("La ventana histórica quedó vacía; revise cobertura de fechas.")
features = observed_orders.groupby("customer_unique_id").agg(
    frequency=("order_id", "nunique"),
    last_purchase=("order_purchase_timestamp", "max"),
)
features["recency_days"] = (cutoff - features.pop("last_purchase")).dt.total_seconds() / 86400

past_order_map = observed_orders[["order_id", "customer_unique_id"]].drop_duplicates()
past_item_rows = items.merge(past_order_map, on="order_id", how="inner")
past_item_rows = past_item_rows.merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
item_features = past_item_rows.groupby("customer_unique_id").agg(
    past_spend_brl=("price", "sum"),
    product_variety=("product_id", "nunique"),
    category_variety=("product_category_name", "nunique"),
)
features = features.join(item_features, how="left").fillna(0)
future_customers = set(future_orders["customer_unique_id"].dropna())
features["repurchased"] = features.index.isin(future_customers).astype(int)
feature_columns = [
    "frequency", "recency_days", "past_spend_brl", "product_variety", "category_variety"
]
X = features[feature_columns]
y = features["repurchased"]
if y.nunique() < 2 or y.value_counts().min() < 2:
    raise ValueError("Se requieren ambas clases y al menos dos clientes por clase para la evaluación.")
print(f"Clientes evaluables: {len(features):,}; fecha de corte: {cutoff.date()}")
display(y.value_counts().rename(index={0: "sin recompra", 1: "con recompra"}).to_frame("clientes"))

## 4. Modelado

Comparamos una referencia que predice la clase más frecuente con Logistic Regression ponderada por clase. Una partición estratificada separa clientes para la evaluación; el corte temporal ya separó correctamente predictores y etiqueta.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
baseline = DummyClassifier(strategy="most_frequent")
classifier = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
)
baseline.fit(X_train, y_train)
classifier.fit(X_train, y_train)
baseline_prediction = baseline.predict(X_test)
prediction = classifier.predict(X_test)

## 5. Evaluación

La matriz de confusión resume aciertos y errores. Precisión mide qué proporción de alertas positivas fue correcta; exhaustividad, qué proporción de recompras detectamos; F1 combina ambas. Con clases desbalanceadas, una referencia mayoritaria puede parecer buena si solo se mira accuracy, por eso reportamos estas métricas.


In [ ]:
def classification_metrics(actual, predicted, model_name):
    return {
        "modelo": model_name,
        "precisión": precision_score(actual, predicted, zero_division=0),
        "exhaustividad": recall_score(actual, predicted, zero_division=0),
        "F1": f1_score(actual, predicted, zero_division=0),
    }

results = pd.DataFrame(
    [
        classification_metrics(y_test, baseline_prediction, "Referencia: clase mayoritaria"),
        classification_metrics(y_test, prediction, "Logistic Regression"),
    ]
)
display(results.round(3))
matrix = confusion_matrix(y_test, prediction, labels=[0, 1])
ConfusionMatrixDisplay(matrix, display_labels=["No", "Sí"]).plot(cmap="Blues")
plt.title("Logistic Regression · recompra en ventana futura")
plt.show()

## 6. Despliegue (demostración temporal)

La demo recibe características agregadas de ejemplo, no identificadores. No se debe usar para tomar decisiones sobre clientes. En Colab, Gradio es temporal y no es un despliegue de producción.


In [ ]:
import gradio as gr

def repurchase_demo(frequency, recency_days, past_spend_brl, product_variety, category_variety):
    example = pd.DataFrame(
        [[frequency, recency_days, past_spend_brl, product_variety, category_variety]],
        columns=feature_columns,
    )
    probability = classifier.predict_proba(example)[0, 1]
    label = "probabilidad estimada de recompra en la ventana de 90 días"
    return f"{label}: {probability:.1%}. Demostración educativa; no usar como decisión real."

demo = gr.Interface(
    fn=repurchase_demo,
    inputs=[
        gr.Number(value=2, minimum=1, label="Pedidos históricos"),
        gr.Number(value=30, minimum=0, label="Días desde última compra"),
        gr.Number(value=150, minimum=0, label="Suma histórica de precios (BRL)"),
        gr.Number(value=2, minimum=1, label="Productos distintos"),
        gr.Number(value=2, minimum=1, label="Categorías distintas"),
    ],
    outputs=gr.Textbox(label="Resultado exploratorio"),
    title="Demo de recompra",
    description="Ventana objetivo de 90 días; entradas de ejemplo, sin identificadores.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)